# Global 모델링 Stage 3 — 사람이 선택한 25개 Feature의 2차 모델 비교

이 Notebook은 사람이 확정한 `global_stage2_selected_25` Feature set만 사용해 Logistic Regression과 XGBoost를 새로 튜닝한다. 1차 `best_params`는 재사용하지 않는다.

- 모델 개발: 고정 Global Train만 사용
- CV: `StratifiedGroupKFold`, 5-fold, group=`SAMPID`, random_state=42
- scoring / threshold: F1 / 0.5
- 전처리: 각 CV Train fold에서만 fit
- Test Dataset, `n_prior_periods`, `sample_weight`: 사용하지 않음

수치와 시각화만 제공한다. Feature set의 적절성이나 최종 모델은 선택하지 않는다.

In [3]:
import importlib
import json
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

ROOT = Path(os.environ.get('KHUDA_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'code').is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError('KHUDA_PROJECT_ROOT에 저장소 루트를 지정하거나 저장소 안에서 Notebook을 실행하세요.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if 'code' in sys.modules and not hasattr(sys.modules['code'], '__path__'):
    del sys.modules['code']
for module_name in ('code.model.first_stage', 'code.pipeline.saved_results', 'code.preprocess.build_features'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from code.model.first_stage import (
    first_stage_summary,
    load_stage_fold_f1,
    run_global_cv_modeling,
    save_modeling_artifacts,
)
from code.model.train import load_model_config
from code.pipeline.saved_results import load_saved_global_train, select_bundle_features
from code.preprocess.build_features import named_feature_set

EXPERIMENT = 'baseline_42features'
FEATURE_SET_NAME = 'global_stage2_selected_25'
RESULT_ROOT = ROOT / 'data' / 'result' / EXPERIMENT
DATASET_PATH = RESULT_ROOT / 'datasets' / 'local_dataset.parquet'
SPLIT_PATH = RESULT_ROOT / 'splits' / 'split_ids.csv'
FEATURE_CONFIG = ROOT / 'code' / 'config' / 'features.yaml'
MODEL_CONFIG = ROOT / 'code' / 'config' / 'model_config.yaml'
STAGE_1_OUTPUT_DIR = RESULT_ROOT / 'modeling' / 'stage_1'
STAGE_3_OUTPUT_DIR = RESULT_ROOT / 'modeling' / 'stage_3_local_group3' # 여기 gruop명 바꾸기

for path in (DATASET_PATH, SPLIT_PATH, FEATURE_CONFIG, MODEL_CONFIG):
    if not path.exists():
        raise FileNotFoundError(f'필요한 Stage 3 입력 파일이 없습니다: {path}')

COLORS = {'stage_1': '#7a7a7a', 'stage_2': '#0066cc'}
LABELS = {'logistic_regression': 'Logistic Regression', 'xgboost': 'XGBoost'}
plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#e0e0e0', 'text.color': '#1d1d1f',
    'axes.labelcolor': '#1d1d1f', 'xtick.color': '#1d1d1f', 'ytick.color': '#1d1d1f',
})

In [4]:
# Test Dataset은 만들지 않는다. 42개 Global Train에서 사람이 확정한 Feature 열만 선택한다.
from code.contracts import DatasetBundle
# 1. 파일 위치를 로컬 데이터셋으로 변경
DATASET_PATH = RESULT_ROOT / 'datasets' / 'local_dataset.parquet'
    
    # 2. 로컬 데이터 전체 불러오기
full_train_bundle = load_saved_global_train(DATASET_PATH, SPLIT_PATH,
  FEATURE_CONFIG)

    # 3. 1번 직군(경영·사무·금융)만 쏙 뽑아내기!
mask = full_train_bundle.metadata['job_group'] == '교육·법률·사회·공공' # 여기 바꾸고
full_train_bundle = DatasetBundle(
        name='local_group1',
        X=full_train_bundle.X[mask].copy(),
        y=full_train_bundle.y[mask].copy(),
        groups=full_train_bundle.groups[mask].copy(),
        metadata=full_train_bundle.metadata[mask].copy(),
        sample_weight=full_train_bundle.sample_weight[mask].copy()
    )

    # 4. 글로벌에서 확정한 '최종 25개 변수' 세팅 적용하기
selected_features = named_feature_set(FEATURE_CONFIG, FEATURE_SET_NAME)
train_bundle = select_bundle_features(
    full_train_bundle, selected_features, name='local_group1_selected_25'
    )
model_config = load_model_config(MODEL_CONFIG)
official_models = model_config['official_comparison_models']
 # 5. 설정 불러오기 및 잘 불러와졌는지 표로 출력
model_config = load_model_config(MODEL_CONFIG)
official_models = model_config['official_comparison_models']


assert official_models == ['logistic_regression', 'xgboost']
assert full_train_bundle.X.shape[1] == 42
assert len(selected_features) == 25
assert list(train_bundle.X.columns) == selected_features
display(pd.DataFrame({'selected_feature_order': range(1, len(selected_features) + 1), 'feature': selected_features}))
display(pd.DataFrame({
    'item': ['Train Person-Periods', 'Train unique SAMPID', 'Stage 1 features', 'Stage 2 features', 'CV', 'scoring', 'threshold'],
    'value': [
        len(train_bundle.y), train_bundle.groups.nunique(), full_train_bundle.X.shape[1], train_bundle.X.shape[1],
        f"StratifiedGroupKFold({model_config['split']['cv_n_splits']}) / SAMPID",
        model_config['tuning']['scoring'], model_config['tuning']['threshold'],
    ],
}))

,selected_feature_order,feature
0,1,gender
1,2,age
2,3,region_5
3,4,baseline_year
4,5,education_level
5,6,student_status
6,7,student_type
7,8,university_type
8,9,major_group
9,10,months_since_graduation


,item,value
0,Train Person-Periods,700
1,Train unique SAMPID,447
2,Stage 1 features,42
3,Stage 2 features,25
4,CV,StratifiedGroupKFold(5) / SAMPID
5,scoring,f1
6,threshold,0.5


## 선택 Feature 기준 재튜닝

아래 셀은 25개 선택 Feature로 LR/XGBoost의 GridSearchCV를 처음부터 실행한다. Stage 1의 `best_params`는 이 단계의 학습 입력으로 사용하지 않는다.

In [6]:
stage_2_results = {}
for model_name in official_models:
    print(f"Running Stage 2 {LABELS[model_name]} ...")
    stage_2_results[model_name] = run_global_cv_modeling(
        train_bundle,
        model_name=model_name,
        feature_config=FEATURE_CONFIG,
        model_config=MODEL_CONFIG,
        # Jupyter의 code 패키지와 process worker 충돌을 피하기 위해 현재 kernel에서 순차 실행한다.
        n_jobs=1,
    )

stage_2_summary = first_stage_summary([stage_2_results[name] for name in official_models])
display(stage_2_summary)
display(pd.DataFrame([
    {
        'model': LABELS[name],
        'best_params': stage_2_results[name].best_params,
        'fold_f1': stage_2_results[name].fold_f1,
    }
    for name in official_models
]))

Running Stage 2 Logistic Regression ...
Running Stage 2 XGBoost ...


ImportError: XGBoost를 사용하려면 code/requirements.txt의 xgboost를 설치하세요.

In [ ]:
# Stage 4 후보 비교에 재사용할 수 있도록 Stage 2 OOF·요약·파라미터·fold F1을 저장한다.
stage_2_paths = save_modeling_artifacts(
    [stage_2_results[name] for name in official_models],
    STAGE_3_OUTPUT_DIR,
    summary_filename='group3_second_stage_summary.csv', # 여기 파일명도 그룹명 바꾸기
)
selected_feature_path = STAGE_3_OUTPUT_DIR / 'selected_features.csv'
pd.DataFrame({'feature_order': range(1, len(selected_features) + 1), 'feature': selected_features}).to_csv(
    selected_feature_path, index=False
)
stage_2_paths['selected_features'] = selected_feature_path
display(pd.DataFrame({'artifact': list(stage_2_paths), 'path': [str(path) for path in stage_2_paths.values()]}))

,artifact,path
0,logistic_regression_oof,C:\Users\didwo\OneDrive\바탕 화면\10th-toy-team1\d...
1,xgboost_oof,C:\Users\didwo\OneDrive\바탕 화면\10th-toy-team1\d...
2,summary,C:\Users\didwo\OneDrive\바탕 화면\10th-toy-team1\d...
3,best_params,C:\Users\didwo\OneDrive\바탕 화면\10th-toy-team1\d...
4,fold_f1,C:\Users\didwo\OneDrive\바탕 화면\10th-toy-team1\d...
5,selected_features,C:\Users\didwo\OneDrive\바탕 화면\10th-toy-team1\d...


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

    # ---------------------------------------------------------
    # 👇 1. 실제 결과물이 저장된 CSV 파일 경로를 문자열로 적어주세요.
    # (본인이 아까 폴더 이름 고치신 경로에 맞게 적어주시면 됩니다)
csv_file_path = stage_2_paths['summary']  # cell 5에서 실제로 저장한 경로를 그대로 재사용 (다른 사람 PC 경로 하드코딩 금지)

    # 👇 2. 차트 제목에 띄울 직군 이름을 적어주세요.
job_group_name = "교육·법률·사회·공공"
    # ---------------------------------------------------------


    # =========================================================
    # 아래는 수정할 필요 없이 그냥 실행하시면 차트가 뜹니다!
    # =========================================================

    # CSV 파일 통째로 읽어오기
df = pd.read_csv(csv_file_path, encoding='utf-8')

    # 데이터 추출
models = [str(m).replace('_', ' ').title() for m in df['model']]
f1_scores = df['oof_f1'].values
roc_aucs = df['oof_roc_auc'].values

    # Apple 디자인 설정
plt.rcParams.update({
        'figure.facecolor': '#f5f5f7',
        'axes.facecolor': '#ffffff',
        'axes.edgecolor': '#e0e0e0',
        'text.color': '#1d1d1f',
        'axes.labelcolor': '#1d1d1f',
        'xtick.color': '#1d1d1f',
        'ytick.color': '#1d1d1f',
        'grid.color': '#f0f0f0'
    })

    # 바 차트 그리기
x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, f1_scores, width, label='F1 Score', color='#0066cc')
rects2 = ax.bar(x + width/2, roc_aucs, width, label='ROC AUC', color='#333333')

ax.set_title(f'[{job_group_name}] 25 Features Performance',
                 fontsize=17, fontweight='600', pad=20, loc='left', fontname ='Malgun Gothic')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=14)
ax.set_ylim(0, 1.0)
ax.grid(axis='y', linestyle='-', linewidth=1)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.legend(frameon=False, loc='upper right', fontsize=12)

for rect in rects1 + rects2:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=12, color='#1d1d1f', fontweight='500')

plt.tight_layout()
plt.show()

## 최종 비교표

이 Notebook은 1차/2차의 Train 내부 CV·OOF 수치와 시각화만 제공한다. 어느 모델을 Stage 4의 Test 평가 대상으로 올릴지는 사람이 직접 결정한다.